In [1]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.layers import (
    Input,
    Embedding,
    Dense,
    MultiHeadAttention,
    LayerNormalization,
    Dropout
)

from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
# English sentences
english = [
    "hello",
    "thank you",
    "good morning",
    "how are you",
    "i am happy",
    "i love you",
    "good night",
    "see you",
    "what is your name",
    "i am fine"
]

# Corresponding French translations
french = [
    "bonjour",
    "merci",
    "bonjour",
    "comment allez vous",
    "je suis heureux",
    "je t aime",
    "bonne nuit",
    "a bientot",
    "comment vous appelez vous",
    "je vais bien"
]

# Add special tokens
french = ["<start> " + s + " <end>" for s in french]
french

['<start> bonjour <end>',
 '<start> merci <end>',
 '<start> bonjour <end>',
 '<start> comment allez vous <end>',
 '<start> je suis heureux <end>',
 '<start> je t aime <end>',
 '<start> bonne nuit <end>',
 '<start> a bientot <end>',
 '<start> comment vous appelez vous <end>',
 '<start> je vais bien <end>']

Tokenization

In [8]:
# English tokenizer
eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(english)

# French tokenizer
fr_tokenizer = Tokenizer(filters="")
fr_tokenizer.fit_on_texts(french)

# Convert sentences into sequences
X = eng_tokenizer.texts_to_sequences(english)
Y = fr_tokenizer.texts_to_sequences(french)

print(X)
print(Y)
# Padding
max_len = max(max(map(len, X)), max(map(len, Y)))

X = pad_sequences(X, maxlen=max_len, padding="post")
Y = pad_sequences(Y, maxlen=max_len, padding="post")
print("x after padding",X)
# Vocabulary sizes
eng_vocab = len(eng_tokenizer.word_index) + 1
fr_vocab = len(fr_tokenizer.word_index) + 1

print("English vocabulary:", eng_vocab)
print("French vocabulary:", fr_vocab)

[[5], [6, 1], [3, 7], [8, 9, 1], [2, 4, 10], [2, 11, 1], [3, 12], [13, 1], [14, 15, 16, 17], [2, 4, 18]]
[[1, 5, 2], [1, 7, 2], [1, 5, 2], [1, 6, 8, 3, 2], [1, 4, 9, 10, 2], [1, 4, 11, 12, 2], [1, 13, 14, 2], [1, 15, 16, 2], [1, 6, 3, 17, 3, 2], [1, 4, 18, 19, 2]]
x after padding [[ 5  0  0  0  0  0]
 [ 6  1  0  0  0  0]
 [ 3  7  0  0  0  0]
 [ 8  9  1  0  0  0]
 [ 2  4 10  0  0  0]
 [ 2 11  1  0  0  0]
 [ 3 12  0  0  0  0]
 [13  1  0  0  0  0]
 [14 15 16 17  0  0]
 [ 2  4 18  0  0  0]]
English vocabulary: 19
French vocabulary: 20


create positional encoding

In [9]:
def positional_encoding(length, depth):

    depth = depth // 2

    positions = np.arange(length)[:, np.newaxis]

    depths = np.arange(depth)[np.newaxis, :] / depth

    angle_rates = 1 / (10000 ** depths)

    angle_rads = positions * angle_rates

    pos_encoding = np.concatenate(
        [np.sin(angle_rads), np.cos(angle_rads)],
        axis=-1
    )

    return tf.cast(pos_encoding, dtype=tf.float32)

Build transformer Encoder


In [10]:
def transformer_encoder(inputs, head_size=64, num_heads=2):

    # Self attention
    attention = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=head_size
    )(inputs, inputs)

    # Residual connection and normalization
    x = LayerNormalization()(inputs + attention)

    # Feed forward network
    ff = Dense(128, activation="relu")(x)
    ff = Dense(64)(ff)

    # Residual connection
    output = LayerNormalization()(x + ff)

    return output

Transformer Decoder

In [12]:
def transformer_decoder(inputs, encoder_output):

    # Masked self attention
    attention1 = MultiHeadAttention(
        num_heads=2,
        key_dim=64
    )(inputs, inputs, use_causal_mask=True)

    x = LayerNormalization()(inputs + attention1)

    # Cross attention
    attention2 = MultiHeadAttention(
        num_heads=2,
        key_dim=64
    )(x, encoder_output)

    x = LayerNormalization()(x + attention2)

    # Feed forward network
    ff = Dense(128, activation="relu")(x)
    ff = Dense(64)(ff)

    output = LayerNormalization()(x + ff)

    return output

Complete transformer

In [20]:
embedding_dim = 64

# Encoder input
encoder_input = Input(shape=(max_len,))

print("encoder input",max_len)
# Decoder input
decoder_input = Input(shape=(max_len-1,))

# Encoder embedding
enc_embedding = Embedding(
    eng_vocab,
    embedding_dim
)(encoder_input)

# Add positional encoding
pos = positional_encoding(max_len, embedding_dim)
dec_pos = positional_encoding(max_len-1, embedding_dim)

enc_embedding = enc_embedding + pos

# Encoder
encoder_output = transformer_encoder(enc_embedding)

# Decoder embedding
dec_embedding = Embedding(
    fr_vocab,
    embedding_dim
)(decoder_input)

# Add positional encoding
dec_embedding = dec_embedding + dec_pos

# Decoder
decoder_output = transformer_decoder(
    dec_embedding,
    encoder_output
)

# Output layer
output = Dense(fr_vocab, activation="softmax")(decoder_output)

# Complete model
model = Model(
    inputs=[encoder_input, decoder_input],
    outputs=output
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

encoder input 6


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_7         │ (None, 6, 64)     │      1,216 │ input_layer_8[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_22 (Add)        │ (None, 6, 64)     │          0 │ embedding_7[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 6, 64)     │     33,216 │ add_22[0][0],     │
│ (MultiHeadAttentio… │                   │            │ add_22[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_9       │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_23 (Add)        │ (None, 6, 64)     │          0 │ add_22[0][0],     │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_8         │ (None, 5, 64)     │      1,280 │ input_layer_9[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 6, 64)     │        128 │ add_23[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_25 (Add)        │ (None, 5, 64)     │          0 │ embedding_8[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 6, 128)    │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 5, 64)     │     33,216 │ add_25[0][0],     │
│ (MultiHeadAttentio… │                   │            │ add_25[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 6, 64)     │      8,256 │ dense_15[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_26 (Add)        │ (None, 5, 64)     │          0 │ add_25[0][0],     │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_24 (Add)        │ (None, 6, 64)     │          0 │ layer_normalizat… │
│                     │                   │            │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 5, 64)     │        128 │ add_26[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 6, 64)     │        128 │ add_24[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 5, 64)     │     33,216 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_27 (Add)        │ (None, 5, 64)     │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten

 Total params: 137,236 (536.08 KB)

 Trainable params: 137,236 (536.08 KB)

 Non-trainable params: 0 (0.00 B)

Training data

In [21]:
# Decoder input: remove the last token
decoder_X = Y[:, :-1]

# Target: remove the first token
decoder_Y = Y[:, 1:]

# Encoder input
encoder_X = X

# Train the model
model.fit(
    [encoder_X, decoder_X],
    decoder_Y,
    epochs=200,
    batch_size=2,
    verbose=1
)

Epoch 1/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.2600 - loss: 3.5391   
Epoch 2/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3600 - loss: 2.3061
Epoch 3/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.2800 - loss: 2.2731
Epoch 4/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.4000 - loss: 2.0994
Epoch 5/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.4200 - loss: 1.9887
Epoch 6/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.4600 - loss: 1.9043
Epoch 7/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.4400 - loss: 1.8021
Epoch 8/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.5000 - loss: 1.7320
Epoch 9/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.4400 - loss: 1.6935
Epoch 10/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.4800 - loss: 1.5839
Epoch 11/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.4800 - loss: 1.5809
Epoch 12/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5400 -

translation function

In [23]:
def translate(sentence):

    # Convert English sentence to sequence
    sequence = eng_tokenizer.texts_to_sequences([sentence])

    sequence = pad_sequences(
        sequence,
        maxlen=max_len,
        padding="post"
    )

    # Initialize decoder with start token
    start_id = fr_tokenizer.word_index["<start>"]

    decoder_input = np.zeros((1, max_len-1), dtype="int32")
    decoder_input[0, 0] = start_id

    result = []

    for i in range(max_len - 1):

        # Predict next word
        predictions = model.predict(
            [sequence, decoder_input],
            verbose=0
        )

        next_id = np.argmax(predictions[0, i])

        word = fr_tokenizer.index_word.get(next_id, "")

        if word == "<end>" or next_id == 0:
            break

        result.append(word)

        decoder_input[0, i + 1] = next_id

    return " ".join(result)

In [24]:
print("English: hello")
print("French:", translate("hello"))

print("\nEnglish: thank you")
print("French:", translate("thank you"))

print("\nEnglish: i love you")
print("French:", translate("i love you"))

English: hello
French: bonjour

English: thank you
French: merci

English: i love you
French: je t aime
